# Qwen SFT marketing với Unsloth (LoRA)

Fine-tune các model dòng Qwen (2B / 4B / 9B …) cho tác vụ viết content marketing tiếng Việt, dùng Unsloth + TRL SFTTrainer.

**Model:** đặt `MODEL_ID` ở ô đầu. Các lựa chọn thông dụng:
- `Qwen/Qwen3.5-2B` · `Qwen/Qwen3.5-4B` · `Qwen/Qwen3.5-9B`
- `unsloth/Qwen3.5-2B` / `unsloth/Qwen3.5-9B` (mirror Unsloth tối ưu)

**VRAM tham khảo (4-bit + LoRA):** 2B ~6 GB · 4B ~10 GB · 9B ~18 GB. Nếu OOM: giảm `PER_DEVICE_BATCH`, tăng `GRAD_ACCUM`, hoặc giảm `MAX_SEQ_LENGTH`.

**Luồng:** cài Unsloth → chuẩn hoá `text` theo ChatML → LoRA SFT → merge + xuất GGUF (mục 5) → so sánh sinh văn trước / sau fine-tune.

**LM Studio:** sau khi có file GGUF, xem hướng dẫn import trong [README.md](../README.md#lm-studio-importing-fine-tuned-models).

## 1. Cấu hình và đường dẫn

In [1]:
import os, gc, platform, random
from pathlib import Path

os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("XFORMERS_DISABLED", "1")
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"

import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel, is_bfloat16_supported

SEED = 3407
random.seed(SEED)
torch.manual_seed(SEED)

# ── Model ──────────────────────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen3.5-9B"  # Đổi: 4B / 9B / unsloth/Qwen3.5-9B …

MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT   = True
DTYPE          = None

# ── Paths ──────────────────────────────────────────────────────────────────────
REPO_ROOT       = Path.cwd().resolve()
MODELS_DIR      = REPO_ROOT / "mcs_train_content_model_outputs" / MODEL_ID.replace("/", "-")
LORA_OUTPUT_DIR = MODELS_DIR / "lora"
DATASET_JSON    = REPO_ROOT / "dataset" / "fnb_dataset_train.json"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

assert DATASET_JSON.is_file(), f"Dataset not found: {DATASET_JSON}"

# ── Training ───────────────────────────────────────────────────────────────────
VAL_FRACTION     = 0.08
NUM_TRAIN_EPOCHS = 2
PER_DEVICE_BATCH = 1
GRAD_ACCUM       = 16
LEARNING_RATE    = 2e-4
DATASET_NUM_PROC = None if platform.system() == "Windows" else 2

assert torch.cuda.is_available(), "Cần GPU + CUDA."
print("CUDA:", torch.cuda.get_device_name(0))
print("MODEL:", MODEL_ID)
print("LORA:", LORA_OUTPUT_DIR)
print("DATA:", DATASET_JSON)

/tmp/ipykernel_2494530/189872858.py:12: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel, is_bfloat16_supported


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
CUDA: NVIDIA GB10
MODEL: Qwen/Qwen3.5-9B
LORA: /home/dc34rpa/nathan/mcs_train_content_model_outputs/Qwen-Qwen3.5-9B/lora
DATA: /home/dc34rpa/nathan/dataset/fnb_dataset_train.json


## 2. Dataset và template ChatML

Ghép `instruction` (system) + `build_user_prompt(title, seed)` (user) + `content` (assistant) theo **ChatML** — cùng format Ollama dùng lúc inference — rồi nối EOS qua `apply_chat_template`.

In [2]:
raw = load_dataset("json", data_files=str(DATASET_JSON))
if "train" in raw:
    full = raw["train"]
else:
    full = raw[list(raw.keys())[0]]

full = full.shuffle(seed=SEED)
n_val = max(1, int(len(full) * VAL_FRACTION))
eval_ds = full.select(range(n_val))
train_ds = full.select(range(n_val, len(full)))
print("train:", len(train_ds), "eval:", len(eval_ds), "columns:", train_ds.column_names)


def build_user_prompt(title: str, seed: str) -> str:
    return (
        f"Viết bài marketing Markdown từ tiêu đề và mồi:\n\n"
        f"Tiêu đề: {title}\n\n"
        f"Mồi:\n{seed}"
    )


def add_text_column(batch, tokenizer):
    texts = []
    for ins, title, seed, content in zip(
        batch["instruction"], batch["title"], batch["seed"], batch["content"]
    ):
        messages = [
            {"role": "system",    "content": ins},
            {"role": "user",      "content": build_user_prompt(title, seed)},
            {"role": "assistant", "content": content},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}


_tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if _tok.pad_token is None:
    _tok.pad_token = _tok.eos_token
EOS_TOKEN = _tok.eos_token
print("EOS_TOKEN repr:", repr(EOS_TOKEN[:40] if len(EOS_TOKEN) > 40 else EOS_TOKEN))

train_tok = train_ds.map(
    lambda b: add_text_column(b, _tok),
    batched=True,
    num_proc=DATASET_NUM_PROC,
    remove_columns=train_ds.column_names,
)
eval_tok = eval_ds.map(
    lambda b: add_text_column(b, _tok),
    batched=True,
    num_proc=DATASET_NUM_PROC,
    remove_columns=eval_ds.column_names,
)
del _tok
gc.collect()

train: 460 eval: 40 columns: ['instruction', 'title', 'seed', 'content', 'id']
EOS_TOKEN repr: '<|im_end|>'


160

## 3. Huấn luyện LoRA (Unsloth + TRL `SFTTrainer`)

In [3]:
def train_lora(model_name: str, output_dir: Path, train_dataset):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=DTYPE,
        load_in_4bit=LOAD_IN_4BIT,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
        use_rslora=False,
        loftq_config=None,
    )

    args = SFTConfig(
        output_dir=str(output_dir / "trainer_output"),
        per_device_train_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=0.1,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,
        report_to="none",
        save_strategy="no",
        padding_free=False,
        max_length=MAX_SEQ_LENGTH,
    )

    _sft_kw = dict(
        model=model,
        train_dataset=train_dataset,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_num_proc=DATASET_NUM_PROC,
        packing=False,
        args=args,
    )
    try:
        trainer = SFTTrainer(tokenizer=tokenizer, **_sft_kw)
    except TypeError:
        trainer = SFTTrainer(processing_class=tokenizer, **_sft_kw)

    trainer.train()
    model.save_pretrained(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    stats = trainer.state.log_history

    del trainer, model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return stats

## 4. Chạy fine-tune

Lưu adapter + tokenizer tại `LORA_OUTPUT_DIR`.

In [4]:
print("=== Train LoRA ===")
_ = train_lora(MODEL_ID, LORA_OUTPUT_DIR, train_tok)
print("Saved:", LORA_OUTPUT_DIR)

=== Train LoRA ===
==((====))==  Unsloth 2026.4.8: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 119.676 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 12.1. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

/home/dc34rpa/nathan/myenv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'dataset_text_field'

## 5. Merge LoRA và xuất (HF 16-bit + GGUF / LM Studio & Ollama)

Chạy sau khi đã train xong và có thư mục `LORA_OUTPUT_DIR`.

- Nạp lại adapter **không** dùng 4-bit, gộp vào base, rồi lưu bản **merged 16-bit** (thư mục `merged_16bit_marketing`) cho Transformers / vLLM — **đây là bản gốc (không quant Q4/Q8) trên stack HF**.
- Xuất **GGUF** vào `gguf` cho **LM Studio** / **Ollama**. Mặc định dùng **`f16` trong GGUF** (trọng số 16-bit, không dùng quant kiểu `q4_k_m`). Đổi `GGUF_QUANT` sang `q4_k_m` (hoặc `q5_k_m`, `q8_0`) nếu cần file nhỏ hơn / vừa VRAM; file `f16` rất lớn với 9B.

Nếu thiếu VRAM khi merge FP16, đặt `SAVE_MERGED_16BIT = False` — vẫn chạy bước xuất GGUF sau đó. Giảm kích thước file GGUF: đổi `GGUF_QUANT` từ `f16` sang `q4_k_m` (hoặc tương tự).

Hướng dẫn import và server: [README.md](../README.md#lm-studio-importing-fine-tuned-models).

In [ ]:
from pathlib import Path
import gc
import torch

assert LORA_OUTPUT_DIR.exists(), f"Chưa có LoRA, hãy chạy ô train trước: {LORA_OUTPUT_DIR}"

MERGED_OUTPUT_DIR = MODELS_DIR / "merged_16bit"
GGUF_OUTPUT_DIR   = MODELS_DIR / "gguf"
MERGED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GGUF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVE_MERGED_16BIT = True
GGUF_QUANT = "f16"

print("Loading LoRA for merge / GGUF...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(LORA_OUTPUT_DIR),
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=False,
)

if SAVE_MERGED_16BIT:
    print("Saving merged 16-bit →", MERGED_OUTPUT_DIR)
    model.save_pretrained_merged(str(MERGED_OUTPUT_DIR), tokenizer, save_method="merged_16bit")

print("Exporting GGUF →", GGUF_OUTPUT_DIR, "| quant =", GGUF_QUANT)
model.save_pretrained_gguf(str(GGUF_OUTPUT_DIR), tokenizer, quantization_method=GGUF_QUANT)

del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print("Xong. GGUF:", GGUF_OUTPUT_DIR)

## 6. So sánh sinh văn: pretrained vs sau SFT

Đổi `EXAMPLE_IDX` để thử mẫu khác trong `eval_ds`.

In [ ]:
# ============================================================
# CELL 6 — So sánh sinh văn (dùng transformers thuần, không Unsloth)
# ============================================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import gc


def build_chat_prompt(tokenizer, instruction: str, title: str, seed: str) -> str:
    messages = [
        {"role": "system", "content": instruction},
        {"role": "user",   "content": build_user_prompt(title, seed)},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


@torch.inference_mode()
def generate_one(model, tokenizer, prompt: str, max_new_tokens: int = 256):
    tok = getattr(tokenizer, "tokenizer", tokenizer)
    input_ids = tok.encode(prompt, return_tensors="pt").to(model.device)
    attention_mask = torch.ones_like(input_ids)

    out = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tok.pad_token_id or tok.eos_token_id,
    )
    generated_ids = out[0][input_ids.shape[-1]:]
    return tok.decode(generated_ids, skip_special_tokens=True).strip()


def load_model_for_inference(model_path: str):
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    return model, tokenizer


COMPARE_VARIANTS = [
    ("Pretrained (chưa SFT marketing)", MODEL_ID, False),
    ("+ LoRA marketing", str(LORA_OUTPUT_DIR), True),
]

EXAMPLE_IDX = 0
row = eval_ds[EXAMPLE_IDX]

print("=" * 80)
print("INSTRUCTION:\n", row["instruction"])
print("\nTITLE:\n", row["title"])
print("\nSEED:\n", row["seed"])
print("\nGOLD RESPONSE:\n", str(row["content"]))
print("=" * 80)

for name, model_path, _is_adapter in COMPARE_VARIANTS:
    print(f"\nLoading {name} ...")
    model, tokenizer = load_model_for_inference(model_path)
    prompt = build_chat_prompt(tokenizer, row["instruction"], row["title"], row["seed"])

    gen = generate_one(model, tokenizer, prompt)
    print(f"\n{'─' * 40}\n>>> {name}\n{'─' * 40}\n{gen}\n")

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

## 7. Prompt tùy chọn (ví dụ F&B tiếng Việt)

In [ ]:
import torch
import gc


@torch.inference_mode()
def generate_one(model, tokenizer, instruction: str, user_input: str, max_new_tokens: int = 256):
    tok = getattr(tokenizer, "tokenizer", tokenizer)
    messages = [
        {"role": "system", "content": instruction},
        {"role": "user",   "content": user_input},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    input_ids = tok.encode(text, return_tensors="pt").to(model.device)
    attention_mask = torch.ones_like(input_ids)

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tok.eos_token_id,
    )
    return tok.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)


CUSTOM_INSTRUCTION = (
    "Xây dựng chiến dịch email marketing có mục tiêu để quảng bá cho buổi ra mắt sản phẩm mới."
)
CUSTOM_INPUT = """\
Company: Cà phê Rang Xay Sài Gòn — thương hiệu cà phê đặc sản nội địa, chuyên dòng single-origin Tây Nguyên
Target Audience: Chủ quán cà phê, F&B manager, người yêu cà phê chuyên nghiệp (25–45 tuổi) tại TP.HCM và Hà Nội
Constraints: Tỷ lệ mở email cao (~30%) nhưng tỷ lệ chuyển đổi thấp (~3%)
Goals: Đạt tỷ lệ mở 35%, tỷ lệ chuyển đổi 12% trong 2 tuần ra mắt
Workflow Stage: Copywriting"""

print("=" * 80)
print("INSTRUCTION:\n", CUSTOM_INSTRUCTION)
print("\nINPUT:\n", CUSTOM_INPUT)
print("=" * 80)

for name, model_path, _ in COMPARE_VARIANTS:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=DTYPE,
        load_in_4bit=LOAD_IN_4BIT,
    )
    FastLanguageModel.for_inference(model)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    gen = generate_one(model, tokenizer, CUSTOM_INSTRUCTION, CUSTOM_INPUT, max_new_tokens=400)
    print(f"\n{'─' * 40}\n>>> {name}\n{'─' * 40}\n{gen}\n")

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()